In [0]:
# ============================================================
# 02_ETL_search.ipynb
# Mục tiêu:
# - Clean keyword
# - Normalize keyword
# - Apply alias rules (optional but recommended)
# - Count + rank deterministic
# - Create most_search_t6, most_search_t7
# ============================================================

from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import unicodedata
import re

In [0]:
# CONFIG
CATALOG = "workspace"
SCHEMA = "customer360"
DB = f"{CATALOG}.{SCHEMA}"

TABLE_LOG_T6 = f"{DB}.logs_t6"
TABLE_LOG_T7 = f"{DB}.logs_t7"
TABLE_RULES = f"{DB}.dim_content_new_rules"

TABLE_MOST_SEARCH_T6 = f"{DB}.new_most_search_t6"
TABLE_MOST_SEARCH_T7 = f"{DB}.new_most_search_t7"

In [0]:
# 
def table_exists(table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(table_name)
    except Exception:
        return False

def normalize_text_py(text: str) -> str:
    """
    Normalize text for alias matching:
    - lower
    - trim
    - collapse spaces
    - remove punctuation
    - remove Vietnamese accents
    """
    if text is None:
        return None

    text = str(text).strip().lower()
    if not text:
        return None

    text = re.sub(r"[\"'`]+", "", text)
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()

    if not text:
        return None

    text_no_accent = unicodedata.normalize("NFD", text)
    text_no_accent = "".join(
        ch for ch in text_no_accent if unicodedata.category(ch) != "Mn"
    )
    text_no_accent = re.sub(r"\s+", " ", text_no_accent).strip()

    return text_no_accent if text_no_accent else None

normalize_text_udf = F.udf(normalize_text_py, StringType())

def clean_keyword(df):
    """
    Clean raw keyword column.
    Assumption: input dataframe has columns user_id, keyword
    """
    return (
        df
        .filter(F.col("keyword").isNotNull())
        .withColumn("keyword", F.trim(F.col("keyword")))
        .filter(F.length("keyword") > 1)
        .withColumn("keyword", F.lower(F.col("keyword")))
        .withColumn("keyword", F.regexp_replace("keyword", r"\s+", " "))
        .filter(F.length("keyword") > 1)
    )

def load_rules():
    """
    Load alias rules if table exists.
    Expected schema:
      alias | canonical_content | category
    Returns normalized alias table:
      alias_norm | canonical_content | category
    """
    if not table_exists(TABLE_RULES):
        print(f"Rules table not found: {TABLE_RULES}")
        return None

    rules_df = (
        spark.table(TABLE_RULES)
        .select("alias", "canonical_content", "category")
        .filter(F.col("alias").isNotNull())
        .filter(F.col("canonical_content").isNotNull())
        .withColumn("alias_norm", normalize_text_udf(F.col("alias")))
        .filter(F.col("alias_norm").isNotNull())
        .select("alias_norm", "canonical_content", "category")
        .dropDuplicates(["alias_norm"])
    )

    return rules_df

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


In [0]:
def process_log_search(df, rules_df=None):
    """
    Transform log search into most_search per user.
    Steps:
    1. Clean keyword
    2. Normalize keyword for alias matching
    3. If rules available -> replace keyword by canonical_content
    4. Count by user_id + keyword
    5. Rank by count desc, keyword asc (deterministic)
    6. Return rank = 1
    """
    df = df.select("user_id", "keyword")
    df = clean_keyword(df)

    # normalize for alias matching
    df = df.withColumn("keyword_norm", normalize_text_udf(F.col("keyword")))
    df = df.filter(F.col("keyword_norm").isNotNull())

    if rules_df is not None:
        df = (
            df.alias("l")
            .join(
                rules_df.alias("r"),
                F.col("l.keyword_norm") == F.col("r.alias_norm"),
                "left"
            )
            .withColumn(
                "keyword_final",
                F.coalesce(F.col("r.canonical_content"), F.col("l.keyword"))
            )
            .select(
                F.col("l.user_id").alias("user_id"),
                F.col("keyword_final").alias("keyword")
            )
        )
    else:
        df = df.select("user_id", "keyword")

    # count
    counted = (
        df.groupBy("user_id", "keyword")
        .count()
        .withColumnRenamed("count", "TotalSearch")
    )

    # deterministic ranking
    window = Window.partitionBy("user_id").orderBy(
        F.col("TotalSearch").desc(),
        F.col("keyword").asc()
    )

    ranked = counted.withColumn("Rank", F.row_number().over(window))

    most_search = (
        ranked
        .filter(F.col("Rank") == 1)
        .select(
            "user_id",
            F.col("keyword").alias("Most_Search"),
            "TotalSearch"
        )
    )

    return most_search


In [0]:
# LOAD SOURCE TABLES
log_t6 = spark.table(TABLE_LOG_T6)
log_t7 = spark.table(TABLE_LOG_T7)


In [0]:
# LOAD RULES 
rules_df = load_rules()
if rules_df is not None:
    print("Loaded rules:", rules_df.count())
    rules_df.show(10, truncate=False)
else:
    print("No rules loaded. ETL will run without alias canonicalization.")


Loaded rules: 16
+-----------------------------+--------------------+------------+
|alias_norm                   |canonical_content   |category    |
+-----------------------------+--------------------+------------+
|u19 viet nam                 |u 19 việt nam       |Sport       |
|con tim sat da               |Con tim sắt đá      |Thai-Drama  |
|vtv                          |vtv                 |News        |
|penthouse                    |penthouse           |K-Drama     |
|2 ngay 1 dem                 |2 ngày 1 đêm        |Reality show|
|demon slayer kimetsu no yaiba|lưỡi gươm diệt quỷ  |Animation   |
|bolero                       |bolero              |Music       |
|thu thach than tuong         |thử thách thần tượng|Reality show|
|taxi driver                  |taxi driver         |Action      |
|truc tiep bong da            |trực tiếp bóng đá   |Sport       |
+-----------------------------+--------------------+------------+
only showing top 10 rows


In [0]:
# PROCESS T6
most_search_t6 = process_log_search(log_t6, rules_df)
most_search_t6.write.mode("overwrite").saveAsTable(TABLE_MOST_SEARCH_T6)

In [0]:
# PROCESS T7
most_search_t7 = process_log_search(log_t7, rules_df)
most_search_t7.write.mode("overwrite").saveAsTable(TABLE_MOST_SEARCH_T7)

In [0]:
# Checking results
print("most_search_t6 users:", spark.table(TABLE_MOST_SEARCH_T6).count())
print("most_search_t7 users:", spark.table(TABLE_MOST_SEARCH_T7).count())

print("\nSample most_search_t6:")
spark.table(TABLE_MOST_SEARCH_T6).show(20, truncate=False)

print("\nSample most_search_t7:")
spark.table(TABLE_MOST_SEARCH_T7).show(20, truncate=False)

most_search_t6 users: 560
most_search_t7 users: 382

Sample most_search_t6:
+--------+-------------------------------+-----------+
|user_id |Most_Search                    |TotalSearch|
+--------+-------------------------------+-----------+
|NULL    |Việt Nam                       |6          |
|0113827 |truyen hinh dong               |1          |
|0162395 |next                           |1          |
|0295684 |hàng xóm của tôi không chịu lớn|1          |
|0467789 |trực tiếp bóng đá              |1          |
|0624390 |bong da viet nam               |1          |
|06405153|him trung ngắn                 |1          |
|06406109|yêu từng centimet              |1          |
|06408144|trực tiếp bóng đá              |1          |
|06408331|bằng chứng thép                |1          |
|06433552|kênh vtv6                      |1          |
|06450113|Việt Nam                       |1          |
|06450121|vn vs afghanistan trực         |1          |
|06450211|xem bóng dá vietnam            |1 